In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run "../0_common/01. env-config"

In [0]:
%run "../0_common/03.silver-helpers"

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.races"
silver_table = f"{catalog_name}.{silver_schema}.races"

In [0]:
races_df = spark.table(bronze_table).filter((F.col("batch_id") == v_batch_id))
display(races_df.head(5))

In [0]:
races_df = races_df.drop("url")

In [0]:
races_df = races_df.withColumnsRenamed(
    {
        "circuitId":"circuit_id",
        "raceName":"race_name",
        "date":"race_date"
    }
)

In [0]:
races_df = races_df.dropDuplicates(["round","season"])

In [0]:
import pyspark.sql.functions as F
races_df = races_df.withColumn(
    "race_name",
    F.initcap(F.col("race_name"))
)

In [0]:
columns_to_update = [col for col in races_df.columns if col not in ["season", "round"]]
columns_to_update

In [0]:
write_to_silver(
    df=races_df,
    target_table=silver_table,
    columns_to_update=columns_to_update,
    merge_condition="t.season = s.season AND t.round = s.round"
)

In [0]:
display(spark.table(silver_table).limit(5))